# Step 3 — 取詞建向量空間
以 N-gram 字元切割 + Chi-square 選取鑑別詞，將每篇文章轉為向量，輸出 `vectors.pkl`

Big Data & Business Analytics, National Taiwan University

In [37]:
# ── 可調整參數區 ──────────────────────────────────────
TOP_K_WORDS        = 200   # 建構向量空間使用的關鍵字數量（各取一半給看漲/看跌）
TERM_SELECT_METHOD = 'chi2'  # 詞彙選取方法：'tf'（依詞頻排序）或 'chi2'（依卡方值排序）
# ──────────────────────────────────────────────────────

In [38]:
import csv
import pickle
from collections import Counter
import re
import monpa
from monpa import utils

# [METHOD] 備用方法：N-gram 字元切割（如需切換請於下方 corpus 建立格取消對應註解）
# def get_ngrams(text, min_n=2, max_n=4):
#     """
#     優化版 N-gram：
#     1. 移除所有空白。
#     2. 只保留中文與英文數字（排除純標點符號）。
#     3. 確保產出的詞彙不含空白。
#     """
#     clean_text = "".join(re.findall(r'[\u4e00-\u9fffA-Za-z0-9]+', text))
#     ngrams = []
#     for n in range(min_n, max_n + 1):
#         for i in range(len(clean_text) - n + 1):
#             gram = clean_text[i:i+n]
#             if not re.search(r'[\u4e00-\u9fffA-Za-z]', gram):
#                 continue
#             ngrams.append(gram)
#     return ngrams

+---------------------------------------------------------------------+
  Welcome to MONPA: Multi-Objective NER POS Annotator for Chinese
+---------------------------------------------------------------------+
已找到 model檔。Found model file.


In [39]:
# 讀入 articles_labeled.csv，分為看漲 / 看跌 兩組
articles = []
f = open('articles_labeled.csv', newline='', encoding='utf-8')
reader = csv.DictReader(f)
for row in reader:
    row['label'] = int(row['label'])
    articles.append(row)
f.close()

bull_arts = [a for a in articles if a['label'] ==  1]
bear_arts = [a for a in articles if a['label'] == -1]
print(f'看漲文章：{len(bull_arts)} 篇')
print(f'看跌文章：{len(bear_arts)} 篇')

看漲文章：1528 篇
看跌文章：1001 篇


In [ ]:
corpus = []   # 每篇文章切完詞後，join 成一個字串
labels = []   # 對應的 label

for i, art in enumerate(articles):
    if i % 200 == 0:
        print(f'{i}/{len(articles)} 篇...')
    if art['label'] == 0:
        continue

    text = art['title'] + ' ' + art['content']

    # [METHOD] 目前方法：monpa 斷詞
    tokens = []
    sentence_list = utils.short_sentence(text)   # 先斷句
    for item in sentence_list:
        result_cut = monpa.cut(item)              # 再斷詞
        for term in result_cut:
            term = term.strip()
            if len(term) > 1:                     # 過濾單字
                tokens.append(term)

    # [METHOD] 備用方法：N-gram 字元切割
    # tokens = get_ngrams(text)

    tokens_str = " ".join(tokens)
    corpus.append(tokens_str)
    labels.append(art['label'])

print(f'處理完成，共 {len(corpus)} 篇文章')

0/4094 篇...
200/4094 篇...
400/4094 篇...
600/4094 篇...
800/4094 篇...
1000/4094 篇...


In [17]:

# 定義輸出的檔名
output_file = 'articles_processed.csv'

# 使用 'w' 模式寫入，並設定 newline='' 避免 Windows 系統出現多餘空行
with open(output_file, 'w', newline='', encoding='utf-8-sig') as f:
    writer = csv.writer(f)
    
    # 寫入標題列 (Header)
    writer.writerow(['processed_text', 'label'])
    
    # 使用 zip 將 corpus 和 labels 兩兩對應寫入
    for text, label in zip(corpus, labels):
        writer.writerow([text, label])

print(f'\n--- 檔案已成功儲存至 {output_file} ---')
print(f'總共儲存了 {len(labels)} 篇有效文章（已排除標籤為 0 者）。')


--- 檔案已成功儲存至 articles_processed.csv ---
總共儲存了 2529 篇有效文章（已排除標籤為 0 者）。


In [ ]:
# import numpy as np

# # ── TF / DF 統計（兩種方法都需要）────────────────────────────
# def compute_tf_df_optimized(corpus_list):
#     tf = Counter()
#     df = Counter()
#     for text in corpus_list:
#         grams = text.split()
#         tf.update(grams)
#         df.update(set(grams))
#     return tf, df

# bull_corpus = [c for c, l in zip(corpus, labels) if l == 1]
# bull_tf, bull_df = compute_tf_df_optimized(bull_corpus)

# bear_corpus = [c for c, l in zip(corpus, labels) if l == -1]
# bear_tf, bear_df = compute_tf_df_optimized(bear_corpus)

# # ── 依 TERM_SELECT_METHOD 選取詞彙並儲存 CSV ─────────────────
# bull_csv = f'top_{TOP_K_WORDS}_bull_terms.csv'
# bear_csv = f'top_{TOP_K_WORDS}_bear_terms.csv'

# if TERM_SELECT_METHOD == 'tf':
#     # [METHOD] 依 TF 詞頻排序，各取 TOP_K_WORDS 個
#     bull_selected = [w for w, _ in bull_tf.most_common(TOP_K_WORDS)]
#     bear_selected = [w for w, _ in bear_tf.most_common(TOP_K_WORDS)]

#     with open(bull_csv, 'w', newline='', encoding='utf-8-sig') as f:
#         writer = csv.writer(f)
#         writer.writerow(['Word', 'TF (總出現次數)', 'DF (出現篇數)'])
#         for w in bull_selected:
#             writer.writerow([w, bull_tf[w], bull_df[w]])
#     print(f'已完成儲存：{bull_csv}（依 TF 排序）')

#     with open(bear_csv, 'w', newline='', encoding='utf-8-sig') as f:
#         writer = csv.writer(f)
#         writer.writerow(['Word', 'TF (總出現次數)', 'DF (出現篇數)'])
#         for w in bear_selected:
#             writer.writerow([w, bear_tf[w], bear_df[w]])
#     print(f'已完成儲存：{bear_csv}（依 TF 排序）')

# elif TERM_SELECT_METHOD == 'chi2':
#     # [METHOD] 依 Chi-square 卡方值排序，使用 sklearn
#     from sklearn.feature_extraction.text import TfidfVectorizer as _TV
#     from sklearn.feature_selection import chi2 as sklearn_chi2

#     # 建立全詞彙 TF 矩陣（chi2 需要非負數值，故 use_idf=False）
#     vec_full = _TV(use_idf=False, lowercase=False, token_pattern=r'\S+')
#     X_full   = vec_full.fit_transform(corpus)
#     vocab    = vec_full.get_feature_names_out()

#     # sklearn chi2 需要 0/1 標籤
#     y_binary = np.array([1 if l == 1 else 0 for l in labels])
#     chi2_scores, chi2_pval = sklearn_chi2(X_full, y_binary)

#     word_chi2 = dict(zip(vocab, chi2_scores))
#     word_pval  = dict(zip(vocab, chi2_pval))

#     # 看漲鑑別詞：在看漲文章 DF 較高的詞，依 chi2 排序取 TOP_K_WORDS 個
#     bull_selected = sorted(
#         [w for w in vocab if bull_df.get(w, 0) > bear_df.get(w, 0)],
#         key=lambda w: word_chi2[w], reverse=True
#     )[:TOP_K_WORDS]

#     # 看跌鑑別詞：在看跌文章 DF 較高的詞，依 chi2 排序取 TOP_K_WORDS 個
#     bear_selected = sorted(
#         [w for w in vocab if bear_df.get(w, 0) >= bull_df.get(w, 0)],
#         key=lambda w: word_chi2[w], reverse=True
#     )[:TOP_K_WORDS]

#     with open(bull_csv, 'w', newline='', encoding='utf-8-sig') as f:
#         writer = csv.writer(f)
#         writer.writerow(['Word', 'Chi2 Score', 'P-value', 'TF (總出現次數)', 'DF (出現篇數)'])
#         for w in bull_selected:
#             writer.writerow([w, round(word_chi2[w], 4), round(word_pval[w], 6),
#                              bull_tf.get(w, 0), bull_df.get(w, 0)])
#     print(f'已完成儲存：{bull_csv}（依 Chi2 排序）')

#     with open(bear_csv, 'w', newline='', encoding='utf-8-sig') as f:
#         writer = csv.writer(f)
#         writer.writerow(['Word', 'Chi2 Score', 'P-value', 'TF (總出現次數)', 'DF (出現篇數)'])
#         for w in bear_selected:
#             writer.writerow([w, round(word_chi2[w], 4), round(word_pval[w], 6),
#                              bear_tf.get(w, 0), bear_df.get(w, 0)])
#     print(f'已完成儲存：{bear_csv}（依 Chi2 排序）')

# print(f'\n--- 統計摘要 ---')
# print(f'看漲選取詞彙：{len(bull_selected)} 個')
# print(f'看跌選取詞彙：{len(bear_selected)} 個')

已完成儲存：top_200_bull_terms.csv（依 Chi2 排序）
已完成儲存：top_200_bear_terms.csv（依 Chi2 排序）

--- 統計摘要 ---
看漲選取詞彙：200 個
看跌選取詞彙：200 個


In [ ]:
# # 依 bull_selected / bear_selected 建立 termset（由上一格產生，對應 TOP_K_WORDS）
# content = corpus

# termset = dict()
# no = 0

# for word in bull_selected:
#     if word not in termset:
#         termset[word] = no
#         no += 1

# for word in bear_selected:
#     if word not in termset:
#         termset[word] = no
#         no += 1

# print(f"termset 建立完成，總共包含 {len(termset)} 個特徵詞。")

termset 建立完成，總共包含 400 個特徵詞。


In [41]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from scipy import sparse
import numpy as np

# # 使用預先的termset參數配置
# vectorizer = TfidfVectorizer(vocabulary=termset, use_idf=True, lowercase=False)

# 執行計算：將內容轉為 TF-IDF 向量
# X = vectorizer.fit_transform(content)

# 步驟一：建 TF-IDF 向量（全部詞）
vectorizer = TfidfVectorizer(use_idf=True, lowercase=False, token_pattern=r'\S+')
X = vectorizer.fit_transform(corpus)
print(f"原始矩陣：{X.shape}")  # (文章數, 全部詞數)


# 步驟二：Chi-square 選出 TOP_K 個最有鑑別力的詞
y = np.array(labels)
selector = SelectKBest(chi2, k=TOP_K_WORDS * 2)  # *2 因為漲跌各一半
X_selected = selector.fit_transform(X, y)
print(f"選後矩陣：{X_selected.shape}")  # (文章數, TOP_K*2)

# 看選出來的詞
termset = vectorizer.get_feature_names_out()[selector.get_support()]
print(f"選出的詞（前20）：{termset[:20]}")

# 寫入整個稀疏矩陣（老師的最後一步）
sparse.save_npz("limit_model.npz", X_selected)

print("向量化與 npz 儲存已完成！")
print(f"矩陣形狀為: {X.shape} (文章數 x 特徵數)")

KeyboardInterrupt: 

In [33]:
import pandas as pd

# 將稀疏矩陣轉為一般矩陣並搭配欄位名稱
df_view = pd.DataFrame(X.toarray(), columns=termset.keys())

# 印出前 5 列看看
print(df_view.head())

         買超  成分股   成分   分股  買超2        資產  前20名  3月營  3月營收  收成長率  ...   沒事  \
0  0.123253  0.0  0.0  0.0  0.0  0.000000   0.0  0.0   0.0   0.0  ...  0.0   
1  0.215720  0.0  0.0  0.0  0.0  0.000000   0.0  0.0   0.0   0.0  ...  0.0   
2  0.000000  0.0  0.0  0.0  0.0  0.611338   0.0  0.0   0.0   0.0  ...  0.0   
3  0.093214  0.0  0.0  0.0  0.0  0.000000   0.0  0.0   0.0   0.0  ...  0.0   
4  0.109002  0.0  0.0  0.0  0.0  0.000000   0.0  0.0   0.0   0.0  ...  0.0   

    及電   凱米  盪個股輪   大期   跌點  台股創  電法說   ou  權重由  
0  0.0  0.0   0.0  0.0  0.0  0.0  0.0  0.0  0.0  
1  0.0  0.0   0.0  0.0  0.0  0.0  0.0  0.0  0.0  
2  0.0  0.0   0.0  0.0  0.0  0.0  0.0  0.0  0.0  
3  0.0  0.0   0.0  0.0  0.0  0.0  0.0  0.0  0.0  
4  0.0  0.0   0.0  0.0  0.0  0.0  0.0  0.0  0.0  

[5 rows x 400 columns]
